In [ ]:
import sys
from pathlib import Path

NOTEBOOK_PATH_CANDIDATES = [Path.cwd(), Path.cwd() / "AgentWorkshop" / "Notebook"]
for candidate in NOTEBOOK_PATH_CANDIDATES:
    if (candidate / "workshop_bootstrap.py").exists():
        resolved_candidate = str(candidate.resolve())
        if resolved_candidate not in sys.path:
            sys.path.insert(0, resolved_candidate)

from workshop_bootstrap import build_workshop_config

CONFIG_OVERRIDES = {
    "resource_group_name": "",
    "location": "",
    "subscription_id": "",
    "foundry_account_name": "",
    "foundry_project_name": "",
    "foundry_project_endpoint": "",
    "foundry_project_api_key": "",
    "search_service_name": "",
    "search_api_key": "",
    "storage_account_name": "",
    "application_insights_name": "",
    "model_zone": "",
}

config = build_workshop_config({k: v for k, v in CONFIG_OVERRIDES.items() if v})
config.show()

# Workshop 5: Multi-Agent Group Chat

This notebook mirrors docs/multi-agent-2.md and creates:
- Orchestrator-Agent
- Data-Analyst-Agent
- Research-Agent
- Recipe-Agent
- Insights-Agent

It also writes a group-chat workflow YAML artifact for the Foundry workflow designer.

In [ ]:
# Uncomment this cell in a clean kernel.
# %pip install --quiet "azure-ai-projects>=2.0.0" azure-identity

In [ ]:
import os
from pathlib import Path

from azure.ai.projects.models import AutoCodeInterpreterToolParam, CodeInterpreterTool, PromptAgentDefinition

from workshop_bootstrap import build_project_client, write_text

if not config.foundry_project_endpoint:
    raise ValueError("Set AZURE_AI_PROJECT_ENDPOINT or provide foundry_project_endpoint in CONFIG_OVERRIDES.")

CHAT_DEPLOYMENT_NAME = os.getenv("CHAT_DEPLOYMENT_NAME", "").strip()
if not CHAT_DEPLOYMENT_NAME:
    raise ValueError("Set CHAT_DEPLOYMENT_NAME in your environment.")

DATASET_GENERAL = Path("../../data/Coffee/CoffeeCSV/GeneralHealth/synthetic_mental_health_dataset.csv").resolve()
DATASET_LARGE = Path("../../data/Coffee/CoffeeCSV/mentalHealth/synthetic_coffee_health_10000.csv").resolve()

project = build_project_client(config)
openai = project.get_openai_client()

with DATASET_GENERAL.open("rb") as file_handle:
    general_file = openai.files.create(purpose="assistants", file=file_handle)
with DATASET_LARGE.open("rb") as file_handle:
    large_file = openai.files.create(purpose="assistants", file=file_handle)

print("Uploaded files for Data-Analyst-Agent:", general_file.id, large_file.id)

In [ ]:
ORCHESTRATOR_AGENT_NAME = "Orchestrator-Agent"
DATA_ANALYST_AGENT_NAME = "Data-Analyst-Agent"
RESEARCH_AGENT_NAME = "Research-Agent"
RECIPE_AGENT_NAME = "Recipe-Agent"
INSIGHTS_AGENT_NAME = "Insights-Agent"
WORKFLOW_NAME = "coffee-group-chat"
WORKFLOW_FILE = Path("../Agent/coffee-group-chat.workflow.yaml").resolve()

orchestrator_prompt = """
You are the Orchestrator Agent.

Your responsibilities:
- Understand user intent.
- Route requests to the right specialist.
- Delegate instead of answering directly.
- Return compressed JSON with next-agent, next-agent-request, and history.
"""

data_analyst_prompt = """
You are the Data Analyst Agent.

Analyze CSV datasets using Code Interpreter.
Return structured summaries with methods, key metrics, uncertainty, and caveats.
"""

research_prompt = """
You are the Research Agent.

Provide scientific insights on coffee and health from trusted sources.
Use clear caveats when evidence is mixed or weak.
"""

recipe_prompt = """
You are the Recipe Agent.

Provide coffee recipes aligned with the user's context and goals.
When relevant, note preparation details that affect caffeine exposure.
"""

insights_prompt = """
You are the Insights Agent.

Synthesize outputs from all other agents into one final answer.
Do not introduce new facts that are not present in upstream agent outputs.
"""

project.agents.create_version(
    agent_name=ORCHESTRATOR_AGENT_NAME,
    definition=PromptAgentDefinition(model=CHAT_DEPLOYMENT_NAME, instructions=orchestrator_prompt, tools=[]),
    description="Routes requests among specialist agents.",
)

project.agents.create_version(
    agent_name=DATA_ANALYST_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=CHAT_DEPLOYMENT_NAME,
        instructions=data_analyst_prompt,
        tools=[
            CodeInterpreterTool(
                container=AutoCodeInterpreterToolParam(file_ids=[general_file.id, large_file.id])
            )
        ],
    ),
    description="Performs quantitative analysis on workshop CSV files.",
)

project.agents.create_version(
    agent_name=RESEARCH_AGENT_NAME,
    definition=PromptAgentDefinition(model=CHAT_DEPLOYMENT_NAME, instructions=research_prompt, tools=[]),
    description="Provides evidence-backed health research context.",
)

project.agents.create_version(
    agent_name=RECIPE_AGENT_NAME,
    definition=PromptAgentDefinition(model=CHAT_DEPLOYMENT_NAME, instructions=recipe_prompt, tools=[]),
    description="Provides coffee recipe and preparation context.",
)

project.agents.create_version(
    agent_name=INSIGHTS_AGENT_NAME,
    definition=PromptAgentDefinition(model=CHAT_DEPLOYMENT_NAME, instructions=insights_prompt, tools=[]),
    description="Synthesizes specialist outputs into final response.",
)

print("Created five group-chat agents.")

In [ ]:
workflow_yaml = f"""
kind: workflow
name: {WORKFLOW_NAME}
description: Group chat multi-agent workflow for coffee workshop
trigger:
  kind: OnConversationStart
  id: trigger_wf
  actions:
    - kind: SetVariable
      id: init_history
      variable: Local.ConvoHistory
      value: =System.LastMessage.Text
    - kind: InvokeAzureAgent
      id: orchestrator_step
      agent:
        name: {ORCHESTRATOR_AGENT_NAME}
      conversationId: =System.ConversationId
      input:
        messages: =Local.ConvoHistory
      output:
        autoSend: true
        responseObject: Local.OrchestratorOutput
        messages: Local.LastOrchestratorMessage
    - kind: SetVariable
      id: next_agent_name
      variable: Local.NextAgentName
      value: =Upper(Trim(Text(Local.OrchestratorOutput.'next-agent')))
    - kind: ConditionGroup
      id: route_group
      conditions:
        - condition: =Local.NextAgentName = 'DATA-ANALYST-AGENT'
          actions:
            - kind: InvokeAzureAgent
              id: data_agent_step
              agent:
                name: {DATA_ANALYST_AGENT_NAME}
              conversationId: =System.ConversationId
              input:
                messages: =Local.LastOrchestratorMessage
              output:
                autoSend: true
                messages: Local.ConvoHistory
            - kind: GotoAction
              id: goto_orchestrator_from_data
              actionId: orchestrator_step
        - condition: =Local.NextAgentName = 'RESEARCH-AGENT'
          actions:
            - kind: InvokeAzureAgent
              id: research_agent_step
              agent:
                name: {RESEARCH_AGENT_NAME}
              conversationId: =System.ConversationId
              input:
                messages: =Local.LastOrchestratorMessage
              output:
                autoSend: true
                messages: Local.ConvoHistory
            - kind: GotoAction
              id: goto_orchestrator_from_research
              actionId: orchestrator_step
        - condition: =Local.NextAgentName = 'RECIPE-AGENT'
          actions:
            - kind: InvokeAzureAgent
              id: recipe_agent_step
              agent:
                name: {RECIPE_AGENT_NAME}
              conversationId: =System.ConversationId
              input:
                messages: =Local.LastOrchestratorMessage
              output:
                autoSend: true
                messages: Local.ConvoHistory
            - kind: GotoAction
              id: goto_orchestrator_from_recipe
              actionId: orchestrator_step
      elseActions:
        - kind: InvokeAzureAgent
          id: insights_step
          agent:
            name: {INSIGHTS_AGENT_NAME}
          conversationId: =System.ConversationId
          input:
            messages: =Local.LastOrchestratorMessage
          output:
            autoSend: true
    - kind: EndConversation
      id: end_conversation
""".strip()

write_text(WORKFLOW_FILE, workflow_yaml)
print(f"Workflow YAML saved to: {WORKFLOW_FILE}")

## Suggested Prompts

- What trends exist between coffee consumption and mental health?
- What does research say about coffee and anxiety?
- Suggest a coffee recipe aligned with reducing stress.